In [12]:
import pandas as pd

In [13]:
# df = pd.read_csv('state_CA.csv', dtype={'county_code': str}, low_memory=False)
# sd = df[(df.county_code == '06073') & (df.action_taken.isin([1, 3]))]
# sd.to_csv('hmda_2025_san_diego.csv', index=False)

In [14]:
df = pd.read_csv("hmda_2025_san_diego.csv", low_memory=False)

In [15]:
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)

In [16]:
df.head().T

,0,1,2,3,4
activity_year,2025,2025,2025,2025,2025
lei,549300LBCBNR1OT00651,549300LBCBNR1OT00651,549300LBCBNR1OT00651,549300LBCBNR1OT00651,549300LBCBNR1OT00651
derived_msa-md,41740,41740,41740,41740,41740
state_code,CA,CA,CA,CA,CA
county_code,6073,6073,6073,6073,6073
census_tract,6073019103.0,6073020602.0,6073010017.0,6073002201.0,6073020025.0
conforming_loan_limit,C,C,C,C,C
derived_loan_product_type,VA:First Lien,Conventional:First Lien,VA:First Lien,VA:First Lien,VA:First Lien
derived_dwelling_category,Single Family (1-4 Units):Site-Built,Single Family (1-4 Units):Site-Built,Single Family (1-4 Units):Site-Built,Single Family (1-4 Units):Site-Built,Single Family (1-4 Units):Site-Built
derived_ethnicity,Joint,Not Hispanic or Latino,Not Hispanic or Latino,Joint,Not Hispanic or Latino


In [17]:
df.columns

Index(['activity_year', 'lei', 'derived_msa-md', 'state_code', 'county_code',
       'census_tract', 'conforming_loan_limit', 'derived_loan_product_type',
       'derived_dwelling_category', 'derived_ethnicity', 'derived_race',
       'derived_sex', 'action_taken', 'purchaser_type', 'preapproval',
       'loan_type', 'loan_purpose', 'lien_status', 'reverse_mortgage',
       'open-end_line_of_credit', 'business_or_commercial_purpose',
       'loan_amount', 'loan_to_value_ratio', 'interest_rate', 'rate_spread',
       'hoepa_status', 'total_loan_costs', 'total_points_and_fees',
       'origination_charges', 'discount_points', 'lender_credits', 'loan_term',
       'prepayment_penalty_term', 'intro_rate_period', 'negative_amortization',
       'interest_only_payment', 'balloon_payment',
       'other_nonamortizing_features', 'property_value', 'construction_method',
       'occupancy_type', 'manufactured_home_secured_property_type',
       'manufactured_home_land_property_interest', 'total_

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68442 entries, 0 to 68441
Data columns (total 99 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   activity_year                             68442 non-null  int64  
 1   lei                                       68442 non-null  object 
 2   derived_msa-md                            68442 non-null  int64  
 3   state_code                                68442 non-null  object 
 4   county_code                               68442 non-null  int64  
 5   census_tract                              68374 non-null  float64
 6   conforming_loan_limit                     67929 non-null  object 
 7   derived_loan_product_type                 68442 non-null  object 
 8   derived_dwelling_category                 68442 non-null  object 
 9   derived_ethnicity                         68442 non-null  object 
 10  derived_race                      

In [19]:
for col in df.columns:
    print(col)
    print(df[col].unique()[:10])
    print()

activity_year
[2025]

lei
['549300LBCBNR1OT00651' '54930039IUN6515TBG78' '549300IRXW9RSZM2B951'
 '254900IRE136HFEI2562' '5493006LGHJCBICSE610' '549300UXTDELQS5SLF91'
 '254900378RFGMBEKAF12' '549300D003QRZFSQ4N43' '6PTKHDJ8HDUF78PFWH30'
 '5493003B2LEGQHDWRN55']

derived_msa-md
[41740]

state_code
['CA']

county_code
[6073]

census_tract
[6.07301910e+09 6.07302060e+09 6.07301002e+09 6.07300220e+09
 6.07302002e+09 6.07301681e+09 6.07301711e+09 6.07301030e+09
 6.07300920e+09 6.07300700e+09]

conforming_loan_limit
['C' 'NC' nan]

derived_loan_product_type
['VA:First Lien' 'Conventional:First Lien' 'FHA:First Lien'
 'Conventional:Subordinate Lien' 'FSA/RHS:First Lien'
 'VA:Subordinate Lien' 'FSA/RHS:Subordinate Lien' 'FHA:Subordinate Lien']

derived_dwelling_category
['Single Family (1-4 Units):Site-Built'
 'Single Family (1-4 Units):Manufactured' 'Multifamily:Site-Built'
 'Multifamily:Manufactured']

derived_ethnicity
['Joint' 'Not Hispanic or Latino' 'Ethnicity Not Available'
 'Hispanic or

In [20]:
df['approved'] = (df.action_taken == 1).astype(int)
df.approved.value_counts(normalize=True)

approved
1    0.767117
0    0.232883
Name: proportion, dtype: float64

In [21]:
leak = ['action_taken', 'purchaser_type', 'interest_rate', 'rate_spread', 'hoepa_status',
        'total_loan_costs', 'total_points_and_fees', 'origination_charges',
        'discount_points', 'lender_credits',
        'denial_reason-1', 'denial_reason-2', 'denial_reason-3', 'denial_reason-4']

constant = ['activity_year', 'state_code', 'county_code', 'derived_msa-md', 'census_tract']

protected = [c for c in df.columns 
             if c.startswith(('derived_', 'applicant', 'co-applicant'))
             and any(w in c for w in ['ethnicity', 'race', 'sex', 'age'])]

fairness = df[protected.copy()]
df = df.drop(columns=leak + constant + protected)
df.shape

(68442, 46)